In [ ]:

import pandas as pd

# 读取训练集
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/bank_customer_churn/train.csv'
train_df = pd.read_csv(train_path)

# 查看数据基本信息
print(train_df.head())
print(train_df.info())
print(train_df.describe())
print(train_df.isnull().sum())


       id  CustomerId     Surname  ...  IsActiveMember EstimatedSalary Exited
0  149380    15780088  Yobachukwu  ...             1.0       103560.98      0
1  164766    15679760    Slattery  ...             0.0       102950.79      0
2  155569    15637678          Ma  ...             0.0       155394.52      0
3  124304    15728693      Galkin  ...             1.0       107428.42      0
4  108008    15613673        Lung  ...             0.0       134110.93      0

[5 rows x 14 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132027 entries, 0 to 132026
Data columns (total 14 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   id               132027 non-null  int64  
 1   CustomerId       132027 non-null  int64  
 2   Surname          132027 non-null  object 
 3   CreditScore      132027 non-null  int64  
 4   Geography        132027 non-null  object 
 5   Gender           132027 non-null  object 
 6   Age              

In [ ]:


import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

# 读取训练集
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/bank_customer_churn/train.csv'
train_df = pd.read_csv(train_path)

# 删除不需要的列
train_df = train_df.drop(columns=['id', 'CustomerId', 'Surname'])

# 定义数值特征和分类特征
numeric_features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
categorical_features = ['Geography', 'Gender']

# 创建预处理管道
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])

# 划分特征和标签
X = train_df.drop(columns=['Exited'])
y = train_df['Exited']

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 预处理数据
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 查看预处理后的数据
print(X_train_processed.shape)
print(X_test_processed.shape)



(105621, 13)
(26406, 13)


In [ ]:


from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score

# 定义模型
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(),
    'Gradient Boosting': GradientBoostingClassifier()
}

# 交叉验证选择最佳模型
best_model = None
best_auc = 0

for name, model in models.items():
    # 交叉验证
    cv_scores = cross_val_score(model, X_train_processed, y_train, cv=5, scoring='roc_auc')
    mean_auc = cv_scores.mean()
    print(f'{name} - Mean AUC: {mean_auc:.4f}')
    
    if mean_auc > best_auc:
        best_auc = mean_auc
        best_model = model

# 使用最佳模型在测试集上进行预测
best_model.fit(X_train_processed, y_train)
y_pred_proba = best_model.predict_proba(X_test_processed)[:, 1]
test_auc = roc_auc_score(y_test, y_pred_proba)

print(f'Best Model: {best_model.__class__.__name__}')
print(f'Test AUC-ROC: {test_auc:.4f}')



Logistic Regression - Mean AUC: 0.8175
Random Forest - Mean AUC: 0.8739
Gradient Boosting - Mean AUC: 0.8879
Best Model: GradientBoostingClassifier
Test AUC-ROC: 0.8875
